### Загрузка данных 

In [1]:
import pandas as pd
# Загружаем данные (разделитель - табуляция, без заголовков)
df = pd.read_csv("SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])

print("Размер данных:", df.shape)
print("\nПервые 5 строк:")
print(df.head())

print("\nРаспределение классов:")
print(df['label'].value_counts())

print("\nВ процентах:")
print(df['label'].value_counts(normalize=True).round(3))

print("\nПропуски:")
print(df.isnull().sum())

Размер данных: (5572, 2)

Первые 5 строк:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Распределение классов:
label
ham     4825
spam     747
Name: count, dtype: int64

В процентах:
label
ham     0.866
spam    0.134
Name: proportion, dtype: float64

Пропуски:
label      0
message    0
dtype: int64


### Функция очистки и превращение ham и spam в метки 0 и 1

In [ ]:
import re 

# 1. Превращение меток в 0/1
df['label_encoded'] = (df['label'] == 'spam').astype(int) 


# 2. Функция очистки
def clean_text(text):
    text = text.lower()                              # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)       # убираем ссылки
    text = re.sub(r'[^a-z\s]', '', text)             # только буквы и пробелы
    text = re.sub(r'\s+', ' ', text).strip()          # лишние пробелы
    return text

df['clean_message'] = df['message'].apply(clean_text)

# 3. Проверка
print("До:", df['message'].iloc[0])
print("После:", df['clean_message'].iloc[0])
print("\nПример спама:")
print(df[df['label'] == 'spam']['clean_message'].iloc[0])
df

До: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
После: go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat

Пример спама:
free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry questionstd txt ratetcs apply overs


,label,message,label_encoded,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",0,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry in a wkly comp to win fa cup final ...
3,ham,U dun say so early hor... U c already then say...,0,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah i dont think he goes to usf he lives aroun...
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1,this is the nd time we have tried contact u u ...
5568,ham,Will ü b going to esplanade fr home?,0,will b going to esplanade fr home
5569,ham,"Pity, * was in mood for that. So...any other s...",0,pity was in mood for that soany other suggestions
5570,ham,The guy did some bitching but I acted like i'd...,0,the guy did some bitching but i acted like id ...


### Разделение на train/val/test с stratify для spam

In [3]:
from sklearn.model_selection import train_test_split

# 1. Сначала отделяем test (20%)
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_encoded']
)

# 2. Потом train/val из оставшихся (25% от 80% = 20% от всех)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42,
    stratify=df_full_train['label_encoded']
)

print(f"Train: {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Val:   {len(df_val)} ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test:  {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)")

print(f"\nСпам в train: {df_train['label_encoded'].mean():.3f}")
print(f"Спам в val:   {df_val['label_encoded'].mean():.3f}")
print(f"Спам в test:  {df_test['label_encoded'].mean():.3f}")

Train: 3342 (60.0%)
Val:   1115 (20.0%)
Test:  1115 (20.0%)

Спам в train: 0.134
Спам в val:   0.135
Спам в test:  0.134


### TF-IDF (униграмма + биграммы) и модель логистической регрессии и class_weight = 'balanced' 

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,          # топ-5000 слов (меньше = быстрее)
    ngram_range=(1, 2),         # униграммы + биграммы ("free prize")
    min_df=2                    # слово должно встречаться хотя бы в 2 документах
)

X_train = vectorizer.fit_transform(df_train['clean_message'])
X_val   = vectorizer.transform(df_val['clean_message'])
X_test  = vectorizer.transform(df_test['clean_message'])

y_train = df_train['label_encoded'].values
y_val   = df_val['label_encoded'].values
y_test  = df_test['label_encoded'].values

print(f"X_train shape: {X_train.shape}")   # (3343, 5000)
print(f"X_val shape:   {X_val.shape}")

# 2. Модель с class_weight='balanced' (важно при дисбалансе!)
model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',    # ← компенсирует дисбаланс 6.5:1
    random_state=42
)

model.fit(X_train, y_train)
print("Модель обучена")

X_train shape: (3342, 5000)
X_val shape:   (1115, 5000)
Модель обучена


a:\sms-scam-detector\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


### Матрица ошибок и Classification report 

In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

# 1. Предсказания на val
y_pred = model.predict(X_val)                    # бинарные (0/1)
y_score = model.predict_proba(X_val)[:, 1]       # вероятности

# 2. Confusion matrix
print("Confusion matrix (val):")
print(confusion_matrix(y_val, y_pred))
# [[TN FP]
#  [FN TP]]

# 3. Метрики
print(f"\nAccuracy : {accuracy_score(y_val, y_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_pred):.4f}")
print(f"Recall   : {recall_score(y_val, y_pred):.4f}")
print(f"F1       : {f1_score(y_val, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val, y_score):.4f}")
print(f"PR AUC   : {average_precision_score(y_val, y_score):.4f}")

print("\nClassification report:")
print(classification_report(y_val, y_pred, target_names=['ham', 'spam']))

Confusion matrix (val):
[[953  12]
 [ 16 134]]

Accuracy : 0.9749
Precision: 0.9178
Recall   : 0.8933
F1       : 0.9054
ROC AUC  : 0.9831
PR AUC   : 0.9615

Classification report:
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       965
        spam       0.92      0.89      0.91       150

    accuracy                           0.97      1115
   macro avg       0.95      0.94      0.95      1115
weighted avg       0.97      0.97      0.97      1115



### Просмотр всех порогов 

In [7]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"{'threshold':<12} {'precision':>10} {'recall':>10} {'f1':>10}")
for t in np.arange(0.1, 0.91, 0.1):
    y_pred_t = (y_score >= t).astype(int)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    f = f1_score(y_val, y_pred_t, zero_division=0)
    print(f"{t:<12.1f} {p:>10.4f} {r:>10.4f} {f:>10.4f}")

threshold     precision     recall         f1
0.1              0.2070     0.9867     0.3422
0.2              0.4712     0.9800     0.6364
0.3              0.7150     0.9533     0.8171
0.4              0.8118     0.9200     0.8625
0.5              0.9178     0.8933     0.9054
0.6              0.9695     0.8467     0.9039
0.7              0.9835     0.7933     0.8782
0.8              1.0000     0.7067     0.8281
0.9              1.0000     0.4533     0.6239


### Финальная модель на train+val → test

In [8]:
# Оптимальным выбран порог 0.4 - компромис по метрикам

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score, confusion_matrix
)

# 1. Финальный TF-IDF на full_train (train + val)
vectorizer_final = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2), min_df=2
)
X_full_train = vectorizer_final.fit_transform(df_full_train['clean_message'])
X_test_final = vectorizer_final.transform(df_test['clean_message'])

# 2. Финальная модель
model_final = LogisticRegression(
    C=1.0, max_iter=1000, class_weight='balanced', random_state=42
)
model_final.fit(X_full_train, df_full_train['label_encoded'].values)

# 3. Test — с выбранным порогом 0.4
THRESHOLD = 0.4
y_test_score = model_final.predict_proba(X_test_final)[:, 1]
y_test_pred = (y_test_score >= THRESHOLD).astype(int)

# 4. Метрики
print("=== FINAL TEST (threshold = 0.4) ===")
print("Confusion matrix:")
print(confusion_matrix(df_test['label_encoded'], y_test_pred))
print()
print(f"Accuracy : {accuracy_score(df_test['label_encoded'], y_test_pred):.4f}")
print(f"Precision: {precision_score(df_test['label_encoded'], y_test_pred):.4f}")
print(f"Recall   : {recall_score(df_test['label_encoded'], y_test_pred):.4f}")
print(f"F1       : {f1_score(df_test['label_encoded'], y_test_pred):.4f}")
print(f"PR AUC   : {average_precision_score(df_test['label_encoded'], y_test_score):.4f}")

=== FINAL TEST (threshold = 0.4) ===
Confusion matrix:
[[941  25]
 [  9 140]]

Accuracy : 0.9695
Precision: 0.8485
Recall   : 0.9396
F1       : 0.8917
PR AUC   : 0.9704


a:\sms-scam-detector\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
